# IGDB API exploratory analysis

This notebook profiles small samples from the IGDB endpoints needed for the first raw ingestion design. It uses the reusable `igdb_pipeline.exploration` helpers, requests at most 100 rows per endpoint, and performs no pagination, GCS writes, or BigQuery writes.

The samples are independently limited and sorted by ID. Relationship coverage below therefore describes **sample overlap**, not referential integrity across the full IGDB dataset.

## Session setup

The repository package is installed from `ingestion/src/igdb_pipeline` in the `ingestion/.venv` environment. One Twitch application token is obtained here and reused for every request in this kernel session. The token value is intentionally never displayed.

In [1]:
import pandas as pd
from IPython.display import display

from igdb_pipeline.auth import get_app_access_token
from igdb_pipeline.config import settings
from igdb_pipeline.exploration import (
    convert_unix_columns,
    field_availability_summary,
    nested_column_summary,
    normalize_records,
    null_summary,
    primary_key_summary,
    query_endpoint,
    relationship_coverage,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

SAMPLE_LIMIT = 100

In [2]:
token = get_app_access_token(
    client_id=settings.twitch_client_id,
    client_secret=settings.twitch_client_secret,
)
access_token = token.access_token

# Confirm authentication without printing the secret token.
print(f"Twitch token acquired for this session; expires in {token.expires_in} seconds.")

Twitch token acquired for this session; expires in 5181995 seconds.


## Requested fields

These field sets are the preliminary recommendation for an initial raw load. They preserve identifiers, names, relationships, source audit timestamps, and high-value descriptive attributes without requesting every IGDB field. `involved_companies` is included as a supporting endpoint because it is the bridge between games and companies.

In [ ]:
ENDPOINT_FIELDS = {
    "games": [
        "id", "name", "slug", "summary", "first_release_date",
        "created_at", "updated_at", "category", "status",
        "rating", "rating_count", "aggregated_rating",
        "aggregated_rating_count", "genres", "platforms",
        "involved_companies", "release_dates", "checksum", "aggregated_rating_count", "aggregated_rating",
    ],
    "platforms": [
        "id", "name", "slug", "abbreviation", "alternative_name",
        "category", "generation", "platform_family", "platform_logo",
        "versions", "websites", "created_at", "updated_at", "checksum",
    ],
    "genres": [
        "id", "name", "slug", "created_at", "updated_at", "checksum",
    ],
    "companies": [
        "id", "name", "slug", "description", "country",
        "developed", "published", "parent", "start_date",
        "changed_company_id", "logo", "websites", "url",
        "created_at", "updated_at", "checksum",
    ],
    "involved_companies": [
        "id", "game", "company", "developer", "publisher",
        "porting", "supporting", "created_at", "updated_at", "checksum",
    ],
    "release_dates": [
        "id", "game", "platform", "date", "human", "m", "y",
        "region", "status", "category", "created_at", "updated_at",
        "checksum",
    ],
}

ENDPOINT_FIELDS

{'games': ['id',
  'name',
  'slug',
  'summary',
  'first_release_date',
  'created_at',
  'updated_at',
  'category',
  'status',
  'rating',
  'rating_count',
  'aggregated_rating',
  'aggregated_rating_count',
  'genres',
  'platforms',
  'involved_companies',
  'release_dates',
  'checksum'],
 'platforms': ['id',
  'name',
  'slug',
  'abbreviation',
  'alternative_name',
  'category',
  'generation',
  'platform_family',
  'platform_logo',
  'versions',
  'websites',
  'created_at',
  'updated_at',
  'checksum'],
 'genres': ['id', 'name', 'slug', 'created_at', 'updated_at', 'checksum'],
 'companies': ['id',
  'name',
  'slug',
  'description',
  'country',
  'developed',
  'published',
  'parent',
  'start_date',
  'changed_company_id',
  'logo',
  'websites',
  'url',
  'created_at',
  'updated_at',
  'checksum'],
 'involved_companies': ['id',
  'game',
  'company',
  'developer',
  'publisher',
  'porting',
  'supporting',
  'created_at',
  'updated_at',
  'checksum'],
 'releas

## Games

Games are the central entity. The sample includes descriptive fields, ratings, lifecycle timestamps, and ID arrays needed to relate games to genres, platforms, company involvement records, and releases.

In [4]:
games = query_endpoint(
    endpoint="games",
    fields=ENDPOINT_FIELDS["games"],
    limit=SAMPLE_LIMIT,
    where="name != null",
    client_id=settings.twitch_client_id,
    access_token=access_token,
)
df_games = normalize_records(games)
df_games = convert_unix_columns(
    df_games, ["first_release_date", "created_at", "updated_at"]
)
print(f"games: {len(df_games)} records")
display(df_games.head())

games: 100 records


,id,aggregated_rating,aggregated_rating_count,created_at,first_release_date,genres,involved_companies,name,platforms,rating,rating_count,release_dates,slug,summary,updated_at,checksum,first_release_date_utc,created_at_utc,updated_at_utc
0,1,90.000000,1.0,1297556438,953596800,"[5, 13, 31]","[2740, 2739, 21014, 265437, 399975, 399976, 399977, 399978, 399979]",Thief II: The Metal Age,[6],85.940605,156.0,"[591513, 591514, 25216, 938194, 938195]",thief-ii-the-metal-age,"The ultimate thief is back! Tread softly as you make your way through 15 new complex, non-linear levels full of loot...",1784058527,aa605627-6f17-a6d9-e861-19e2ab192bcb,2000-03-21 00:00:00+00:00,2011-02-13 00:20:38+00:00,2026-07-14 19:48:47+00:00
1,2,70.000000,1.0,1297561968,912384000,"[13, 31]","[7, 6, 265431]",Thief: The Dark Project,[6],86.407010,176.0,"[25214, 591506]",thief-the-dark-project,"Thief is a first-person stealth game that likes the dark. You sneak through the ruins of haunted cathedrals, subterr...",1781036550,cd635271-4ad6-8d55-e993-1869bb157305,1998-11-30 00:00:00+00:00,2011-02-13 01:52:48+00:00,2026-06-09 20:22:30+00:00
2,3,83.000000,2.0,1297562649,1085443200,"[5, 13, 31]","[489, 490, 21015, 265438]",Thief: Deadly Shadows,"[11, 6]",81.708697,133.0,"[25220, 25223, 25219, 25221, 25222]",thief-deadly-shadows,"In the third instalment of the Thief series, master-thief Garrett is contacted by Keeper Artemus with a mission to s...",1783135026,3b143938-e84f-b6de-34b2-ed800b8b7438,2004-05-25 00:00:00+00:00,2011-02-13 02:04:09+00:00,2026-07-04 03:17:06+00:00
3,4,63.642857,14.0,1297562870,1393286400,"[5, 31]","[265439, 265440, 265441, 265442, 265443]",Thief,"[9, 48, 6, 14, 12, 49]",69.903715,341.0,"[106370, 106371, 136391, 591525, 591531, 591530, 591528, 591526, 591529, 591532, 591527, 591534, 591535, 591536, 591...",thief,"There is a rising tide of fear in The City. Hatred saturates every stone and whilst the rich prosper, the less fortu...",1783106038,3560f06b-eecd-480d-44fe-3de22fec83c6,2014-02-25 00:00:00+00:00,2011-02-13 02:07:50+00:00,2026-07-03 19:13:58+00:00
4,5,NaN,NaN,1297562972,914198400,[12],"[11, 13, 18471, 12]",Baldur's Gate,"[3, 6, 39, 14]",85.255797,349.0,"[28327, 149982, 178907, 799603, 799604, 799605]",baldur-s-gate,Baldur's Gate is a fantasy role-playing video game developed by BioWare and published in 1998 by Interplay Entertain...,1784042195,1be32526-32c2-cdfc-74a7-e0df259f8fb0,1998-12-21 00:00:00+00:00,2011-02-13 02:09:32+00:00,2026-07-14 15:16:35+00:00


In [5]:
display(primary_key_summary(df_games))
display(field_availability_summary(df_games, ENDPOINT_FIELDS["games"]))
display(null_summary(df_games))
display(nested_column_summary(df_games))

,key,row_count,null_count,duplicate_count,distinct_count,is_valid
0,id,100,0,0,100,True


,requested_field,is_returned
0,id,True
1,name,True
2,slug,True
3,summary,True
4,first_release_date,True
5,created_at,True
6,updated_at,True
7,category,False
8,status,False
9,rating,True


,null_percentage
aggregated_rating_count,36.0
aggregated_rating,36.0
rating,2.0
rating_count,2.0
release_dates,0.0
created_at_utc,0.0
first_release_date_utc,0.0
checksum,0.0
updated_at,0.0
summary,0.0


,column,pandas_dtype,observed_types,contains_list,contains_dict,example
0,id,int64,int,False,False,1
1,aggregated_rating,float64,float,False,False,90.0
2,aggregated_rating_count,float64,float,False,False,1.0
3,created_at,int64,int,False,False,1297556438
4,first_release_date,int64,int,False,False,953596800
5,genres,object,list,True,False,"[5, 13, 31]"
6,involved_companies,object,list,True,False,"[2740, 2739, 21014, 265437, 399975, 399976, 399977, 399978, 399979]"
7,name,str,str,False,False,Thief II: The Metal Age
8,platforms,object,list,True,False,[6]
9,rating,float64,float,False,False,85.940605


### Games observations

- `id` is the expected primary key and is checked for nulls and duplicates above.
- `genres`, `platforms`, `involved_companies`, and `release_dates` are list-valued relationship fields. They should remain intact in the raw layer; downstream models can unnest them into bridges.
- `first_release_date`, `created_at`, and `updated_at` are Unix-second fields; the added `_utc` columns make their meaning visible without replacing the raw values.
- Ratings and summaries are optional and may be sparse. `updated_at` is the strongest candidate watermark for incremental game extraction.

## Platforms

Platforms provide the reference dimension for `games.platforms[]` and `release_dates.platform`. Arrays such as versions and websites are retained for raw fidelity.

In [6]:
platforms = query_endpoint(
    endpoint="platforms",
    fields=ENDPOINT_FIELDS["platforms"],
    limit=SAMPLE_LIMIT,
    client_id=settings.twitch_client_id,
    access_token=access_token,
)
df_platforms = normalize_records(platforms)
df_platforms = convert_unix_columns(df_platforms, ["created_at", "updated_at"])
print(f"platforms: {len(df_platforms)} records")
display(df_platforms.head())
display(primary_key_summary(df_platforms))
display(
    field_availability_summary(df_platforms, ENDPOINT_FIELDS["platforms"])
)
display(null_summary(df_platforms))
display(nested_column_summary(df_platforms))

platforms: 100 records


,id,abbreviation,alternative_name,created_at,name,platform_logo,platform_family,slug,updated_at,versions,websites,checksum,generation,created_at_utc,updated_at_utc
0,3,Linux,GNU/Linux,1297639288,Linux,380.0,4.0,linux,1718027289,[44],[1],c211a78a-f57c-507c-d715-2aa72f0553f1,NaN,2011-02-13 23:21:28+00:00,2024-06-10 13:48:09+00:00
1,4,N64,N64,1297639288,Nintendo 64,260.0,5.0,n64,1718027289,"[122, 51]",[35],bdd70b18-e2cb-0d2e-fc8b-6d5d08daee20,5.0,2011-02-13 23:21:28+00:00,2024-06-10 13:48:09+00:00
2,5,Wii,Revolution,1297639288,Wii,326.0,5.0,wii,1718027289,"[283, 730, 731, 72]",[275],874eb898-0730-6912-1b3d-1faa56ac318d,7.0,2011-02-13 23:21:28+00:00,2024-06-10 13:48:09+00:00
3,6,PC,mswin,1297639288,PC (Microsoft Windows),670.0,NaN,win,1718027289,"[530, 14, 1, 15, 13, 534, 531, 532, 513, 529, 124, 533]",[2],aa236f37-9001-de17-45ea-141fa09d6605,NaN,2011-02-13 23:21:28+00:00,2024-06-10 13:48:09+00:00
4,7,PS1,"PSX, PSOne, PS",1297639288,PlayStation,803.0,1.0,ps,1718027289,"[57, 653, 654]",NaN,54b6c8e8-54c2-b887-b668-e44425cca7a6,5.0,2011-02-13 23:21:28+00:00,2024-06-10 13:48:09+00:00


,key,row_count,null_count,duplicate_count,distinct_count,is_valid
0,id,100,0,0,100,True


,requested_field,is_returned
0,id,True
1,name,True
2,slug,True
3,abbreviation,True
4,alternative_name,True
5,category,False
6,generation,True
7,platform_family,True
8,platform_logo,True
9,versions,True


,null_percentage
websites,73.0
platform_family,65.0
generation,50.0
alternative_name,40.0
platform_logo,19.0
abbreviation,1.0
id,0.0
created_at,0.0
name,0.0
slug,0.0


,column,pandas_dtype,observed_types,contains_list,contains_dict,example
0,id,int64,int,False,False,3
1,abbreviation,str,str,False,False,Linux
2,alternative_name,str,str,False,False,GNU/Linux
3,created_at,int64,int,False,False,1297639288
4,name,str,str,False,False,Linux
5,platform_logo,float64,float,False,False,380.0
6,platform_family,float64,float,False,False,4.0
7,slug,str,str,False,False,linux
8,updated_at,int64,int,False,False,1718027289
9,versions,object,list,True,False,[44]


### Platforms observations

- `id` is the expected primary key. `platform_family` and `platform_logo` are optional scalar references, while `versions` and `websites` are arrays.
- Alternative names, abbreviations, generation, and family membership are not guaranteed for every platform.
- `updated_at` can support incremental ingestion; raw arrays should be preserved and modeled separately only when an analytical use case requires them.

## Genres

Genres are a compact reference dimension used by the `games.genres[]` ID array.

In [7]:
genres = query_endpoint(
    endpoint="genres",
    fields=ENDPOINT_FIELDS["genres"],
    limit=SAMPLE_LIMIT,
    client_id=settings.twitch_client_id,
    access_token=access_token,
)
df_genres = normalize_records(genres)
df_genres = convert_unix_columns(df_genres, ["created_at", "updated_at"])
print(f"genres: {len(df_genres)} records")
display(df_genres.head())
display(primary_key_summary(df_genres))
display(field_availability_summary(df_genres, ENDPOINT_FIELDS["genres"]))
display(null_summary(df_genres))
display(nested_column_summary(df_genres))

genres: 23 records


,id,created_at,name,slug,updated_at,checksum,created_at_utc,updated_at_utc
0,2,1297639288,Point-and-click,point-and-click,1323382086,b295f28a-5f68-fc3e-5de2-f3195e10d160,2011-02-13 23:21:28+00:00,2011-12-08 22:08:06+00:00
1,4,1297639288,Fighting,fighting,1323289215,d23da988-5bb7-011e-34dc-0b712765e470,2011-02-13 23:21:28+00:00,2011-12-07 20:20:15+00:00
2,5,1297639288,Shooter,shooter,1323289215,20c255bc-ec2d-c219-34bd-47995b0a79f1,2011-02-13 23:21:28+00:00,2011-12-07 20:20:15+00:00
3,7,1297639288,Music,music,1323289215,cbf12469-a5a8-6974-086c-ec9c6aeb5791,2011-02-13 23:21:28+00:00,2011-12-07 20:20:15+00:00
4,8,1297639288,Platform,platform,1323289215,06c188b9-1058-933f-dfee-4ec658117ad2,2011-02-13 23:21:28+00:00,2011-12-07 20:20:15+00:00


,key,row_count,null_count,duplicate_count,distinct_count,is_valid
0,id,23,0,0,23,True


,requested_field,is_returned
0,id,True
1,name,True
2,slug,True
3,created_at,True
4,updated_at,True
5,checksum,True


,null_percentage
id,0.0
created_at,0.0
name,0.0
slug,0.0
updated_at,0.0
checksum,0.0
created_at_utc,0.0
updated_at_utc,0.0


,column,pandas_dtype,observed_types,contains_list,contains_dict,example
0,id,int64,int,False,False,2
1,created_at,int64,int,False,False,1297639288
2,name,str,str,False,False,Point-and-click
3,slug,str,str,False,False,point-and-click
4,updated_at,int64,int,False,False,1323382086
5,checksum,str,str,False,False,b295f28a-5f68-fc3e-5de2-f3195e10d160
6,created_at_utc,"datetime64[s, UTC]",Timestamp,False,False,2011-02-13 23:21:28+00:00
7,updated_at_utc,"datetime64[s, UTC]",Timestamp,False,False,2011-12-08 22:08:06+00:00


### Genres observations

- `id` is the expected primary key; `name` and `slug` are the useful descriptive attributes.
- This endpoint has no expected array-valued fields in the requested schema.
- `updated_at` is available for change tracking, although this small reference table could also be refreshed in full.

## Companies

Companies contain organization attributes, but they do **not** directly resolve a game's role-specific company relationships. The `involved_companies` endpoint is required to connect a game to a company and identify whether the company was a developer, publisher, porter, or supporter.

In [8]:
companies = query_endpoint(
    endpoint="companies",
    fields=ENDPOINT_FIELDS["companies"],
    limit=SAMPLE_LIMIT,
    client_id=settings.twitch_client_id,
    access_token=access_token,
)
df_companies = normalize_records(companies)
df_companies = convert_unix_columns(
    df_companies, ["start_date", "created_at", "updated_at"]
)
print(f"companies: {len(df_companies)} records")
display(df_companies.head())
display(primary_key_summary(df_companies))
display(
    field_availability_summary(df_companies, ENDPOINT_FIELDS["companies"])
)
display(null_summary(df_companies))
display(nested_column_summary(df_companies))

companies: 100 records


,id,changed_company_id,country,created_at,description,developed,logo,name,published,slug,start_date,updated_at,url,websites,checksum,parent,start_date_utc,created_at_utc,updated_at_utc
0,1,5.0,840.0,1297810753,"Electronic Arts is a leading global interactive entertainment software company. EA delivers games, content and onlin...","[3632, 5053, 5530, 5363, 5396, 6211, 1621, 8100, 4588, 18705, 8813, 12714, 4429, 4394, 19772, 79594, 13838, 26013, 2...",20818.0,Electronic Arts,"[1274, 1832, 1059, 1268, 74, 1310, 1314, 585, 321, 2249, 3553, 349, 240, 336, 335, 76, 1216, 191847, 1306, 3126, 390...",electronic-arts,391392000.0,1783832808,https://www.igdb.com/companies/electronic-arts,[1076],2b177699-711f-010e-c859-dca7f1c6151d,NaN,1982-05-28 00:00:00+00:00,2011-02-15 22:59:13+00:00,2026-07-12 05:06:48+00:00
1,2,NaN,124.0,1297810806,"BioWare is a Canadian video game developer founded by newly graduated medical doctors Ray Muzyka, Greg Zeschuk and A...","[74, 73, 5, 213, 81, 217, 218, 13912, 13910, 13690, 77200, 45181, 20188, 20189, 17465, 17464, 17462, 376722, 22963, ...",6325.0,BioWare,"[22963, 22975, 124906, 25299, 13784, 141859, 10]",bioware,791596800.0,1771330855,https://www.igdb.com/companies/bioware,[1057],03282bf3-0c37-bbf8-d12d-4edbfa82cb67,1.0,1995-02-01 00:00:00+00:00,2011-02-15 23:00:06+00:00,2026-02-17 12:20:55+00:00
2,3,NaN,840.0,1297817573,Looking Glass Studios was a Massachusetts-based game developer that operated from 1990 to 2000. Originally founded a...,"[2, 70, 22, 715, 714, 1, 10686, 12987, 3352, 47735, 62333, 47738, 9227, 12908, 23, 647]",13.0,Looking Glass Studios,"[70, 3352, 62333]",looking-glass-studios,638841600.0,1783536350,https://www.igdb.com/companies/looking-glass-studios,[16500],bd7733b0-be7d-6002-385d-67888a1a41ea,NaN,1990-03-31 00:00:00+00:00,2011-02-16 00:52:53+00:00,2026-07-08 18:45:50+00:00
3,4,4940.0,826.0,1297817598,Now called Square Enix Europe. The company was founded as Domark in 1984 by Mark Strachan and Dominic Wheatley. In 1...,"[5770, 5259, 4223, 120295, 70113, 81432, 286664, 275496, 73152, 43735, 70258]",357.0,Eidos Interactive,"[3823, 2143, 3201, 1870, 2, 42, 243, 3, 275, 714, 8471, 968, 1050, 1042, 6233, 1, 477, 653, 748, 4192, 3713, 3959, 6...",eidos-interactive,833500800.0,1782227243,https://www.igdb.com/companies/eidos-interactive,[358],34fec21a-9b2e-9d49-d551-2dc2516d0b0a,NaN,1996-05-31 00:00:00+00:00,2011-02-16 00:53:18+00:00,2026-06-23 15:07:23+00:00
4,5,NaN,840.0,1297817647,"Founded in 1983, Interplay Entertainment is a developer, publisher and licensor of video game software headquartered...","[3461, 253, 15152, 45878, 18153, 12509, 15010, 15011, 130816, 130687, 38819, 775, 308359, 13, 77192, 94418, 36423, 3...",955.0,Interplay Entertainment,"[2220, 3461, 3479, 722, 5, 655, 669, 3950, 2402, 129, 6433, 8565, 220, 253, 8829, 2503, 18682, 11780, 19566, 10911, ...",interplay-entertainment,441676800.0,1783670252,https://www.igdb.com/companies/interplay-entertainment,[1036],892bb858-c630-6f72-207a-82a7c53ddae1,NaN,1983-12-31 00:00:00+00:00,2011-02-16 00:54:07+00:00,2026-07-10 07:57:32+00:00


,key,row_count,null_count,duplicate_count,distinct_count,is_valid
0,id,100,0,0,100,True


,requested_field,is_returned
0,id,True
1,name,True
2,slug,True
3,description,True
4,country,True
5,developed,True
6,published,True
7,parent,True
8,start_date,True
9,changed_company_id,True


,null_percentage
changed_company_id,86.0
parent,52.0
websites,39.0
published,31.0
description,26.0
logo,16.0
start_date_utc,16.0
start_date,16.0
country,13.0
developed,7.0


,column,pandas_dtype,observed_types,contains_list,contains_dict,example
0,id,int64,int,False,False,1
1,changed_company_id,float64,float,False,False,5.0
2,country,float64,float,False,False,840.0
3,created_at,int64,int,False,False,1297810753
4,description,str,str,False,False,"Electronic Arts is a leading global interactive entertainment software company. EA delivers games, content and onlin..."
5,developed,object,list,True,False,"[3632, 5053, 5530, 5363, 5396, 6211, 1621, 8100, 4588, 18705, 8813, 12714, 4429, 4394, 19772, 79594, 13838, 26013, 2..."
6,logo,float64,float,False,False,20818.0
7,name,str,str,False,False,Electronic Arts
8,published,object,list,True,False,"[1274, 1832, 1059, 1268, 74, 1310, 1314, 585, 321, 2249, 3553, 349, 240, 336, 335, 76, 1216, 191847, 1306, 3126, 390..."
9,slug,str,str,False,False,electronic-arts


### Companies observations

- `id` is the expected primary key. `parent` and `changed_company_id` are optional self-references to `companies.id`.
- `developed`, `published`, and `websites` are arrays. They are useful source context, but the authoritative role-bearing game/company bridge is `involved_companies`.
- Country, description, start date, parent, and logo may be null. `updated_at` is the preferred incremental field.

## Involved companies (supporting relationship endpoint)

This bridge endpoint resolves `games.involved_companies[]`. Each record identifies one game/company association and carries Boolean role flags.

In [9]:
involved_companies = query_endpoint(
    endpoint="involved_companies",
    fields=ENDPOINT_FIELDS["involved_companies"],
    limit=SAMPLE_LIMIT,
    client_id=settings.twitch_client_id,
    access_token=access_token,
)
df_involved_companies = normalize_records(involved_companies)
df_involved_companies = convert_unix_columns(
    df_involved_companies, ["created_at", "updated_at"]
)
print(f"involved_companies: {len(df_involved_companies)} records")
display(df_involved_companies.head())
display(primary_key_summary(df_involved_companies))
display(
    field_availability_summary(
        df_involved_companies, ENDPOINT_FIELDS["involved_companies"]
    )
)
display(null_summary(df_involved_companies))
display(nested_column_summary(df_involved_companies))

involved_companies: 100 records


,id,company,created_at,developer,game,porting,publisher,supporting,updated_at,checksum,created_at_utc,updated_at_utc
0,2,1,1298851200,False,38,False,True,False,1321056000,869e47f6-88ff-28a8-dabd-30c21aeea7e5,2011-02-28 00:00:00+00:00,2011-11-12 00:00:00+00:00
1,6,3,1298937600,True,2,False,False,False,1321056000,daadcb53-ce9e-b449-d0b1-a4b87560a5be,2011-03-01 00:00:00+00:00,2011-11-12 00:00:00+00:00
2,7,4,1298937600,False,2,False,True,False,1321056000,12622471-47ba-c241-c506-6d3211a913ab,2011-03-01 00:00:00+00:00,2011-11-12 00:00:00+00:00
3,8,7,1298937600,True,38,False,False,False,1321056000,a3d3730c-932c-30d3-25ab-12f2343e242e,2011-03-01 00:00:00+00:00,2011-11-12 00:00:00+00:00
4,9,11,1298937600,True,37,False,False,False,1321056000,14b13028-299d-23d0-e3ed-acd02a55d96d,2011-03-01 00:00:00+00:00,2011-11-12 00:00:00+00:00


,key,row_count,null_count,duplicate_count,distinct_count,is_valid
0,id,100,0,0,100,True


,requested_field,is_returned
0,id,True
1,game,True
2,company,True
3,developer,True
4,publisher,True
5,porting,True
6,supporting,True
7,created_at,True
8,updated_at,True
9,checksum,True


,null_percentage
id,0.0
company,0.0
created_at,0.0
developer,0.0
game,0.0
porting,0.0
publisher,0.0
supporting,0.0
updated_at,0.0
checksum,0.0


,column,pandas_dtype,observed_types,contains_list,contains_dict,example
0,id,int64,int,False,False,2
1,company,int64,int,False,False,1
2,created_at,int64,int,False,False,1298851200
3,developer,bool,bool,False,False,False
4,game,int64,int,False,False,38
5,porting,bool,bool,False,False,False
6,publisher,bool,bool,False,False,True
7,supporting,bool,bool,False,False,False
8,updated_at,int64,int,False,False,1321056000
9,checksum,str,str,False,False,869e47f6-88ff-28a8-dabd-30c21aeea7e5


### Involved-companies observations

- `id` is the bridge record primary key. `game -> games.id` and `company -> companies.id` are expected foreign keys.
- Developer, publisher, porting, and supporting flags describe the company's role for that game.
- The bridge should be loaded as its own raw table; deriving relationships only from the company arrays would lose role semantics.

## Release dates

Release-date records connect a game and platform to a date, region, and date precision/category. Multiple records per game are expected, so `game` is not unique even though the release-date `id` should be.

In [10]:
release_dates = query_endpoint(
    endpoint="release_dates",
    fields=ENDPOINT_FIELDS["release_dates"],
    limit=SAMPLE_LIMIT,
    client_id=settings.twitch_client_id,
    access_token=access_token,
)
df_release_dates = normalize_records(release_dates)
df_release_dates = convert_unix_columns(
    df_release_dates, ["date", "created_at", "updated_at"]
)
print(f"release_dates: {len(df_release_dates)} records")
display(df_release_dates.head())
display(primary_key_summary(df_release_dates))
display(
    field_availability_summary(
        df_release_dates, ENDPOINT_FIELDS["release_dates"]
    )
)
display(null_summary(df_release_dates))
display(nested_column_summary(df_release_dates))

release_dates: 100 records


,id,created_at,date,game,human,m,platform,updated_at,y,checksum,date_utc,created_at_utc,updated_at_utc
0,4,1298900219,1224460800,37,"Oct 20, 2008",10,6,1757431002,2008,26e9dc8c-23ee-e7bc-0345-8c3a4cf27dd1,2008-10-20 00:00:00+00:00,2011-02-28 13:36:59+00:00,2025-09-09 15:16:42+00:00
1,8,1299320105,1223942400,37,"Oct 14, 2008",10,9,1757431002,2008,4ddfbbcb-1118-0762-0d4c-b402a1234510,2008-10-14 00:00:00+00:00,2011-03-05 10:15:05+00:00,2025-09-09 15:16:42+00:00
2,9,1299320153,1223942400,37,"Oct 14, 2008",10,12,1757431002,2008,dcdfc703-f44e-8133-54ac-5e53080869cc,2008-10-14 00:00:00+00:00,2011-03-05 10:15:53+00:00,2025-09-09 15:16:42+00:00
3,46,1300109716,1226966400,122,"Nov 18, 2008",11,6,1757144078,2008,a704106c-9030-c73c-d414-2029a0142f90,2008-11-18 00:00:00+00:00,2011-03-14 13:35:16+00:00,2025-09-06 07:34:38+00:00
4,48,1300110586,1101168000,123,"Nov 23, 2004",11,14,1757007648,2004,597c4e9b-f7b1-d7df-be5b-51855c2460b6,2004-11-23 00:00:00+00:00,2011-03-14 13:49:46+00:00,2025-09-04 17:40:48+00:00


,key,row_count,null_count,duplicate_count,distinct_count,is_valid
0,id,100,0,0,100,True


,requested_field,is_returned
0,id,True
1,game,True
2,platform,True
3,date,True
4,human,True
5,m,True
6,y,True
7,region,False
8,status,False
9,category,False


,null_percentage
id,0.0
created_at,0.0
date,0.0
game,0.0
human,0.0
m,0.0
platform,0.0
updated_at,0.0
y,0.0
checksum,0.0


,column,pandas_dtype,observed_types,contains_list,contains_dict,example
0,id,int64,int,False,False,4
1,created_at,int64,int,False,False,1298900219
2,date,int64,int,False,False,1224460800
3,game,int64,int,False,False,37
4,human,str,str,False,False,"Oct 20, 2008"
5,m,int64,int,False,False,10
6,platform,int64,int,False,False,6
7,updated_at,int64,int,False,False,1757431002
8,y,int64,int,False,False,2008
9,checksum,str,str,False,False,26e9dc8c-23ee-e7bc-0345-8c3a4cf27dd1


### Release-date observations

- `id` is the expected primary key; `game` and `platform` are expected foreign keys.
- `date`, `created_at`, and `updated_at` are Unix seconds. `human` is display text, while `m`, `y`, and `category` describe partial/estimated date information.
- Region and status/category fields may be null or represented by IGDB enum IDs. Preserve the raw values and decode enums in curated models. `updated_at` is the incremental watermark candidate.

## Sampled relationship coverage

The checks below explicitly test the expected ID relationships. Because every endpoint is independently capped at 100 rows, an unmatched ID normally means the referenced record fell outside the target sample—not that IGDB contains a broken foreign key. Full integrity checks belong after paginated raw extraction.

In [11]:
relationship_checks = pd.concat(
    [
        relationship_coverage(df_games, "genres", df_genres),
        relationship_coverage(df_games, "platforms", df_platforms),
        relationship_coverage(
            df_games, "involved_companies", df_involved_companies
        ),
        relationship_coverage(df_involved_companies, "game", df_games),
        relationship_coverage(
            df_involved_companies, "company", df_companies
        ),
        relationship_coverage(df_release_dates, "game", df_games),
        relationship_coverage(df_release_dates, "platform", df_platforms),
    ],
    ignore_index=True,
)
relationship_checks.insert(
    0,
    "expected_relationship",
    [
        "games.genres[] -> genres.id",
        "games.platforms[] -> platforms.id",
        "games.involved_companies[] -> involved_companies.id",
        "involved_companies.game -> games.id",
        "involved_companies.company -> companies.id",
        "release_dates.game -> games.id",
        "release_dates.platform -> platforms.id",
    ],
)
display(relationship_checks)

,expected_relationship,relationship,reference_count,unique_reference_count,matched_unique_count,unmatched_unique_count,sample_coverage_percentage
0,games.genres[] -> genres.id,genres -> id,236,15,15,0,100.00
1,games.platforms[] -> platforms.id,platforms -> id,322,30,26,4,86.67
2,games.involved_companies[] -> involved_companies.id,involved_companies -> id,332,332,17,315,5.12
3,involved_companies.game -> games.id,game -> id,100,58,10,48,17.24
4,involved_companies.company -> companies.id,company -> id,100,61,45,16,73.77
5,release_dates.game -> games.id,game -> id,100,64,9,55,14.06
6,release_dates.platform -> platforms.id,platform -> id,100,14,14,0,100.00


### Relationship observations

The core model is `games` at the center, with genre and platform many-to-many relationships stored as ID arrays. `games.involved_companies[] -> involved_companies.id` first resolves bridge records, and each bridge then resolves `company -> companies.id`. Releases independently reference both games and platforms. Coverage percentages must be interpreted only as overlap among these deterministic 100-row samples.

## Endpoint record counts

In [12]:
endpoint_record_counts = pd.DataFrame(
    {
        "endpoint": [
            "games", "platforms", "genres", "companies",
            "involved_companies", "release_dates",
        ],
        "record_count": [
            len(df_games), len(df_platforms), len(df_genres), len(df_companies),
            len(df_involved_companies), len(df_release_dates),
        ],
        "sample_limit": SAMPLE_LIMIT,
    }
)
display(endpoint_record_counts)

,endpoint,record_count,sample_limit
0,games,100,100
1,platforms,100,100
2,genres,23,100
3,companies,100,100
4,involved_companies,100,100
5,release_dates,100,100


## Exploration Conclusions

### Proposed raw tables

- `raw_games`
- `raw_platforms`
- `raw_genres`
- `raw_companies`
- `raw_involved_companies`
- `raw_release_dates`

Each table should preserve the source payload fields and add ingestion metadata such as `_ingested_at`, `_batch_id`, and optionally `_source_extracted_at`. No cloud writes are performed by this notebook.

### Recommended initial fields by endpoint

- **games:** `id`, `name`, `slug`, `summary`, `first_release_date`, `created_at`, `updated_at`, `category`, `status`, rating fields, `genres`, `platforms`, `involved_companies`, `release_dates`, `checksum`.
- **platforms:** `id`, names/slugs, abbreviation, category, generation, `platform_family`, `platform_logo`, `versions`, `websites`, source timestamps, `checksum`.
- **genres:** `id`, `name`, `slug`, source timestamps, `checksum`.
- **companies:** `id`, names/slugs, description, country, `developed`, `published`, `parent`, `start_date`, `changed_company_id`, `logo`, `websites`, URL, source timestamps, `checksum`.
- **involved_companies:** `id`, `game`, `company`, all role flags, source timestamps, `checksum`.
- **release_dates:** `id`, `game`, `platform`, `date`, `human`, month/year, region, status/category, source timestamps, `checksum`.

The executable `ENDPOINT_FIELDS` mapping above is the exact requested-field specification.

### Arrays and nested fields

- Games: `genres`, `platforms`, `involved_companies`, `release_dates`.
- Platforms: `versions`, `websites`; `platform_family` and `platform_logo` are scalar nested references returned as IDs when requested without expansion.
- Companies: `developed`, `published`, `websites`; `logo`, `parent`, and `changed_company_id` are scalar references.
- Genres, involved companies, and release dates have no expected arrays in this field selection.

Raw BigQuery schemas can use repeated scalar IDs for relationship arrays. Curated bridge tables should be created later for game-genre and game-platform relationships. Game-company relationships must use `involved_companies` so the role flags are retained.

### Expected primary and foreign keys

- Primary key for every proposed raw table: `id`.
- `games.genres[] -> genres.id`.
- `games.platforms[] -> platforms.id`.
- `games.involved_companies[] -> involved_companies.id`.
- `involved_companies.game -> games.id`; `involved_companies.company -> companies.id`.
- `release_dates.game -> games.id`; `release_dates.platform -> platforms.id`.
- Optional company self-references: `companies.parent -> companies.id` and `companies.changed_company_id -> companies.id`.

### Data quality concerns

- Optional descriptions, ratings, dates, region/status values, abbreviations, family/logo references, and company metadata create substantial nulls. IGDB omits absent properties rather than always returning explicit nulls, so a requested field can be missing from the sample DataFrame entirely; production schemas must not be inferred from one sample.
- Relationship arrays may be absent, empty, or contain IDs outside a limited sample. Sample coverage is not a full referential-integrity result.
- Enum-like category, status, and region fields arrive as numeric IDs and need documented decoding downstream.
- Unix seconds must be interpreted as UTC; invalid values should be quarantined or converted to null with an observable quality check.
- Company arrays overlap conceptually with the involved-company bridge but do not carry equivalent role semantics.
- Source records can change, be merged, or disappear; `changed_company_id`, `checksum`, extraction metadata, and snapshot retention help audit those changes.

### Fields useful for incremental ingestion

Use `updated_at` as the main source watermark for all six endpoints and retain `created_at` for audit and backfill checks. `checksum` can help detect payload changes. `first_release_date`, `start_date`, and release `date` are business dates, not reliable ingestion watermarks. Incremental extraction needs a deterministic `(updated_at, id)` cursor to avoid ties.

### Next technical steps

1. Add pagination and retry-aware extraction in the ingestion package, not in this notebook.
2. Define versioned raw schemas that preserve arrays and source timestamps plus ingestion metadata.
3. Extract complete endpoint snapshots to GCS, then load the six proposed raw BigQuery tables.
4. Implement incremental `(updated_at, id)` watermarks with overlap/lookback handling and deduplication.
5. Build curated dimensions and bridge tables by unnesting game arrays and resolving involved-company roles.
6. Run full-dataset uniqueness, referential-integrity, enum-domain, timestamp-range, and freshness tests.